In [0]:
import os, sys
sys.path.append(os.path.abspath('..'))

from src.config import PARES_AWESOME, SECRET_SCOPE
from src.extrator import coletar_json, finalizar
from src.http_client import criar_sessao

dbutils.widgets.dropdown('modo', 'intradiario', ['intradiario', 'diario', 'backfill'])
modo = dbutils.widgets.get('modo')

BASE = 'https://economia.awesomeapi.com.br/json'
sessao = criar_sessao()
headers = {'x-api-key': dbutils.secrets.get(SECRET_SCOPE, 'awesomeapi_token')}
logs = []

if modo == 'intradiario':
    pares = ','.join(PARES_AWESOME)
    logs.append(coletar_json(sessao, 'awesomeapi', 'cotacoes_atuais', f'{BASE}/last/{pares}', headers))

if modo in ('diario', 'backfill'):
    dias = 5 if modo == 'diario' else 360
    logs.append(coletar_json(sessao, 'awesomeapi', 'catalogo_pares', f'{BASE}/available', headers))
    for par in PARES_AWESOME:
        logs.append(coletar_json(sessao, 'awesomeapi', 'historico_diario', f'{BASE}/daily/{par}/{dias}', headers))

finalizar(logs)